<a href="https://www.kaggle.com/code/tay208/fully-supervised-train-on-30-train-dataset?scriptVersionId=302171846" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pwd

/kaggle/working


In [2]:
!cp -r /kaggle/input/polyppvt/Polyp-PVT/utils /kaggle/working/utils
!cp -r /kaggle/input/polyppvt/Polyp-PVT/lib /kaggle/working/lib
!cp -r /kaggle/input/polyppvt/Polyp-PVT/pretrained_pth /kaggle/working/pretrained_pth
!cp -r /kaggle/input/polyppvt/Polyp-PVT/dataset /kaggle/working/dataset

In [3]:
%%writefile /kaggle/working/train_mean_teacher.py
import argparse
import copy
import logging
import os
from datetime import datetime
from itertools import cycle

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from lib.pvt import PolypPVT
from utils.dataloader import get_semi_loaders, test_dataset
from utils.utils import AvgMeter, adjust_lr, clip_gradient


def structure_loss(pred, mask):
    weit = 1 + 5 * torch.abs(F.avg_pool2d(mask, kernel_size=31, stride=1, padding=15) - mask)
    wbce = F.binary_cross_entropy_with_logits(pred, mask, reduction="none")
    wbce = (weit * wbce).sum(dim=(2, 3)) / weit.sum(dim=(2, 3))

    pred = torch.sigmoid(pred)
    inter = ((pred * mask) * weit).sum(dim=(2, 3))
    union = ((pred + mask) * weit).sum(dim=(2, 3))
    wiou = 1 - (inter + 1) / (union - inter + 1)

    return (wbce + wiou).mean()


def consistency_loss(student_pred, teacher_pred):
    student_prob = torch.sigmoid(student_pred)
    teacher_prob = torch.sigmoid(teacher_pred)
    return F.mse_loss(student_prob, teacher_prob)


def get_consistency_weight(epoch, max_weight, rampup_epoch):
    if rampup_epoch <= 0:
        return max_weight
    cur = np.clip(epoch, 0.0, rampup_epoch)
    phase = 1.0 - cur / rampup_epoch
    return max_weight * float(np.exp(-5.0 * phase * phase))


@torch.no_grad()
def update_ema_variables(student_model, teacher_model, ema_decay, global_step):
    # Warm up EMA to avoid unstable teacher targets at the beginning.
    alpha = min(1.0 - 1.0 / (global_step + 1), ema_decay)
    for teacher_param, student_param in zip(teacher_model.parameters(), student_model.parameters()):
        teacher_param.data.mul_(alpha).add_(student_param.data, alpha=1.0 - alpha)


@torch.no_grad()
def test(model, path, dataset, test_size):
    data_path = os.path.join(path, dataset)
    image_root = f"{data_path}/images/"
    gt_root = f"{data_path}/masks/"

    model.eval()
    num_samples = len(os.listdir(gt_root))
    loader = test_dataset(image_root, gt_root, test_size)
    dsc_sum = 0.0

    for _ in range(num_samples):
        image, gt, _ = loader.load_data()
        gt = np.asarray(gt, np.float32)
        gt /= (gt.max() + 1e-8)
        image = image.cuda()

        pred1, pred2 = model(image)
        pred = F.interpolate(pred1 + pred2, size=gt.shape, mode="bilinear", align_corners=False)
        pred = pred.sigmoid().data.cpu().numpy().squeeze()
        pred = (pred - pred.min()) / (pred.max() - pred.min() + 1e-8)

        pred_flat = np.reshape(pred, (-1))
        gt_flat = np.reshape(gt, (-1))
        intersection = pred_flat * gt_flat
        smooth = 1
        dice = (2 * intersection.sum() + smooth) / (pred.sum() + gt.sum() + smooth)
        dsc_sum += float(f"{dice:.4f}")

    return dsc_sum / num_samples


def make_views(images, student_noise_std, teacher_noise_std):
    s_noise = torch.clamp(torch.randn_like(images) * student_noise_std, -2 * student_noise_std, 2 * student_noise_std)
    t_noise = torch.clamp(torch.randn_like(images) * teacher_noise_std, -2 * teacher_noise_std, 2 * teacher_noise_std)
    return images + s_noise, images + t_noise


def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def train_one_epoch(labeled_loader, unlabeled_loader, student_model, teacher_model, optimizer, scaler, epoch, opt, total_step, global_step):
    student_model.train()
    teacher_model.eval()

    size_rates = [0.75, 1, 1.25]
    sup_meter = AvgMeter()
    cons_meter = AvgMeter()
    total_meter = AvgMeter()

    consistency_w = get_consistency_weight(epoch, opt.consistency_weight, opt.consistency_rampup)

    unlabeled_iter = cycle(unlabeled_loader) if unlabeled_loader is not None else None

    for step, pack in enumerate(labeled_loader, start=1):
        images_l, gts_l = pack
        images_l = images_l.cuda(non_blocking=True)
        gts_l = gts_l.cuda(non_blocking=True)

        images_u = None
        if unlabeled_iter is not None:
            images_u = next(unlabeled_iter).cuda(non_blocking=True)

        for rate in size_rates:
            optimizer.zero_grad()

            train_size = int(round(opt.trainsize * rate / 32) * 32)
            if rate != 1:
                images_l_r = F.interpolate(images_l, size=(train_size, train_size), mode="bilinear", align_corners=True)
                gts_l_r = F.interpolate(gts_l, size=(train_size, train_size), mode="bilinear", align_corners=True)
                images_u_r = None
                if images_u is not None:
                    images_u_r = F.interpolate(images_u, size=(train_size, train_size), mode="bilinear", align_corners=True)
            else:
                images_l_r = images_l
                gts_l_r = gts_l
                images_u_r = images_u

            with torch.cuda.amp.autocast(enabled=opt.use_amp):
                s_pred1_l, s_pred2_l = student_model(images_l_r)
                sup_loss = structure_loss(s_pred1_l, gts_l_r) + structure_loss(s_pred2_l, gts_l_r)

                if images_u_r is not None:
                    student_in_u, teacher_in_u = make_views(images_u_r, opt.student_noise_std, opt.teacher_noise_std)
                    s_pred1_u, s_pred2_u = student_model(student_in_u)
                    with torch.no_grad():
                        t_pred1_u, t_pred2_u = teacher_model(teacher_in_u)
                    cons_loss = consistency_loss(s_pred1_u, t_pred1_u) + consistency_loss(s_pred2_u, t_pred2_u)
                else:
                    cons_loss = torch.tensor(0.0, device=images_l_r.device)

                total_loss = sup_loss + consistency_w * cons_loss

            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            clip_gradient(optimizer, opt.clip)
            scaler.step(optimizer)
            scaler.update()

            global_step += 1
            update_ema_variables(student_model, teacher_model, opt.ema_decay, global_step)

            if rate == 1:
                sup_meter.update(sup_loss.data, opt.batchsize)
                cons_meter.update(cons_loss.data, opt.batchsize)
                total_meter.update(total_loss.data, opt.batchsize)

        if step % 20 == 0 or step == total_step:
            print(
                "{} Epoch [{:03d}/{:03d}], Step [{:04d}/{:04d}], sup: {:0.4f}, cons: {:0.4f}, total: {:0.4f}, w: {:0.4f}".format(
                    datetime.now(),
                    epoch,
                    opt.epoch,
                    step,
                    total_step,
                    sup_meter.show(),
                    cons_meter.show(),
                    total_meter.show(),
                    consistency_w,
                )
            )

    return global_step


def validate_and_checkpoint(teacher_model, epoch, opt, best_score, dict_plot):
    save_path = opt.train_save
    if not os.path.exists(save_path):
        os.makedirs(save_path)

    teacher_state = unwrap_model(teacher_model).state_dict()
    torch.save(teacher_state, os.path.join(save_path, f"{epoch}_PolypPVT_teacher.pth"))

    test_path = "./dataset/TestDataset/"
    for dataset in ["CVC-300", "CVC-ClinicDB", "Kvasir", "CVC-ColonDB", "ETIS-LaribPolypDB"]:
        dataset_dice = test(teacher_model, test_path, dataset, opt.trainsize)
        logging.info("epoch: %s, dataset: %s, dice: %s", epoch, dataset, dataset_dice)
        print(dataset, ": ", dataset_dice)
        dict_plot[dataset].append(dataset_dice)

    mean_dice = test(teacher_model, opt.test_path, "test", opt.trainsize)
    dict_plot["test"].append(mean_dice)

    if mean_dice > best_score:
        best_score = mean_dice
        torch.save(teacher_state, os.path.join(save_path, "PolypPVT_teacher_best.pth"))
        torch.save(teacher_state, os.path.join(save_path, f"{epoch}_PolypPVT_teacher_best.pth"))
        print("##############################################################################best", best_score)
        logging.info("##############################################################################best:%s", best_score)

    return best_score


if __name__ == "__main__":
    dict_plot = {
        "CVC-300": [],
        "CVC-ClinicDB": [],
        "Kvasir": [],
        "CVC-ColonDB": [],
        "ETIS-LaribPolypDB": [],
        "test": [],
    }

    model_name = "PolypPVT_MeanTeacher"

    parser = argparse.ArgumentParser()
    parser.add_argument("--epoch", type=int, default=100, help="epoch number")
    parser.add_argument("--lr", type=float, default=1e-4, help="learning rate")
    parser.add_argument("--optimizer", type=str, default="AdamW", help="choosing optimizer AdamW or SGD")
    parser.add_argument("--augmentation", default=False, help="choose to do random flip rotation")
    parser.add_argument("--batchsize", type=int, default=16, help="training batch size")
    parser.add_argument("--trainsize", type=int, default=352, help="training dataset size")
    parser.add_argument("--clip", type=float, default=0.5, help="gradient clipping margin")
    parser.add_argument("--decay_rate", type=float, default=0.1, help="decay rate of learning rate")
    parser.add_argument("--decay_epoch", type=int, default=50, help="every n epochs decay learning rate")
    parser.add_argument("--train_path", type=str, default="./dataset/TrainDataset/", help="path to train dataset")
    parser.add_argument("--test_path", type=str, default="./dataset/TestDataset/", help="path to testing dataset")
    parser.add_argument("--train_save", type=str, default="./model_pth/" + model_name + "/")
    parser.add_argument("--num_workers", type=int, default=8, help="dataloader workers")
    parser.add_argument("--pin_memory", default=True, help="use pin_memory for dataloader")
    parser.add_argument("--labeled_ratio", type=float, default=0.5, help="ratio of training data used as labeled")
    parser.add_argument("--split_seed", type=int, default=42, help="random seed for labeled/unlabeled split")
    parser.add_argument("--unlabeled_batchsize", type=int, default=0, help="batch size for unlabeled loader; <=0 means use --batchsize")

    # Mean Teacher specific arguments
    parser.add_argument("--ema_decay", type=float, default=0.99, help="EMA decay for teacher updates")
    parser.add_argument("--consistency_weight", type=float, default=1.0, help="max consistency loss weight")
    parser.add_argument("--consistency_rampup", type=int, default=30, help="ramp-up epochs for consistency loss")
    parser.add_argument("--student_noise_std", type=float, default=0.10, help="student input Gaussian noise std")
    parser.add_argument("--teacher_noise_std", type=float, default=0.05, help="teacher input Gaussian noise std")
    parser.add_argument("--use_amp", default=True, help="enable automatic mixed precision")

    opt = parser.parse_args()
    opt.use_amp = str(opt.use_amp).lower() == "true"
    opt.pin_memory = str(opt.pin_memory).lower() == "true"

    torch.backends.cudnn.benchmark = True

    logging.basicConfig(
        filename="train_log.log",
        format="[%(asctime)s-%(filename)s-%(levelname)s:%(message)s]",
        level=logging.INFO,
        filemode="a",
        datefmt="%Y-%m-%d %I:%M:%S %p",
    )

    student_model = PolypPVT().cuda()
    teacher_model = copy.deepcopy(student_model).cuda()
    teacher_model.eval()
    for param in teacher_model.parameters():
        param.requires_grad = False

    gpu_count = torch.cuda.device_count()
    if gpu_count > 1:
        print(f"Using DataParallel on {gpu_count} GPUs")
        student_model = nn.DataParallel(student_model)
        teacher_model = nn.DataParallel(teacher_model)

    if opt.optimizer == "AdamW":
        optimizer = torch.optim.AdamW(student_model.parameters(), opt.lr, weight_decay=1e-4)
    else:
        optimizer = torch.optim.SGD(student_model.parameters(), opt.lr, weight_decay=1e-4, momentum=0.9)
    scaler = torch.cuda.amp.GradScaler(enabled=opt.use_amp)

    print(optimizer)

    image_root = f"{opt.train_path}/images/"
    gt_root = f"{opt.train_path}/masks/"

    if opt.unlabeled_batchsize <= 0:
        opt.unlabeled_batchsize = opt.batchsize

    labeled_loader, unlabeled_loader, split_info = get_semi_loaders(
        image_root,
        gt_root,
        labeled_batchsize=opt.batchsize,
        unlabeled_batchsize=opt.unlabeled_batchsize,
        trainsize=opt.trainsize,
        labeled_ratio=opt.labeled_ratio,
        split_seed=opt.split_seed,
        num_workers=opt.num_workers,
        pin_memory=opt.pin_memory,
        augmentation=opt.augmentation,
    )

    total_step = len(labeled_loader)

    print(
        "Semi-supervised split -> total: {}, labeled: {}, unlabeled: {}, labeled_ratio: {}".format(
            split_info["total_count"],
            split_info["labeled_count"],
            split_info["unlabeled_count"],
            opt.labeled_ratio,
        )
    )

    best = 0.0
    global_step = 0

    print("#" * 20, "Start Mean Teacher Training", "#" * 20)

    for epoch in range(1, opt.epoch):
        adjust_lr(optimizer, opt.lr, epoch, opt.decay_rate, opt.decay_epoch)
        global_step = train_one_epoch(
            labeled_loader,
            unlabeled_loader,
            student_model,
            teacher_model,
            optimizer,
            scaler,
            epoch,
            opt,
            total_step,
            global_step,
        )
        best = validate_and_checkpoint(teacher_model, epoch, opt, best, dict_plot)


Writing /kaggle/working/train_mean_teacher.py


In [4]:
%%writefile /kaggle/working/utils/dataloader.py
import os
from PIL import Image
import torch.utils.data as data
import torchvision.transforms as transforms
import numpy as np
import random
import torch


def _is_image_file(name):
    return name.endswith('.jpg') or name.endswith('.png')


def _is_mask_file(name):
    return name.endswith('.png') or name.endswith('.jpg') or name.endswith('.tif')


def _build_image_transform(trainsize, augmentations):
    if augmentations == 'True':
        return transforms.Compose([
            transforms.RandomRotation(90),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.Resize((trainsize, trainsize)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    return transforms.Compose([
        transforms.Resize((trainsize, trainsize)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


def _build_mask_transform(trainsize, augmentations):
    if augmentations == 'True':
        return transforms.Compose([
            transforms.RandomRotation(90),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.Resize((trainsize, trainsize)),
            transforms.ToTensor(),
        ])
    return transforms.Compose([
        transforms.Resize((trainsize, trainsize)),
        transforms.ToTensor(),
    ])


class PolypDataset(data.Dataset):
    """
    dataloader for polyp segmentation tasks
    """
    def __init__(self, image_root, gt_root, trainsize, augmentations, image_paths=None, gt_paths=None):
        self.trainsize = trainsize
        self.augmentations = augmentations
        print(self.augmentations)
        if image_paths is None:
            self.images = [os.path.join(image_root, f) for f in os.listdir(image_root) if _is_image_file(f)]
        else:
            self.images = list(image_paths)

        if gt_paths is None:
            self.gts = [os.path.join(gt_root, f) for f in os.listdir(gt_root) if _is_mask_file(f)]
        else:
            self.gts = list(gt_paths)

        self.images = sorted(self.images)
        self.gts = sorted(self.gts)
        self.filter_files()
        self.size = len(self.images)
        if self.augmentations == 'True':
            print('Using RandomRotation, RandomFlip')
        else:
            print('no augmentation')
        self.img_transform = _build_image_transform(self.trainsize, self.augmentations)
        self.gt_transform = _build_mask_transform(self.trainsize, self.augmentations)
            

    def __getitem__(self, index):
        
        image = self.rgb_loader(self.images[index])
        gt = self.binary_loader(self.gts[index])
        
        seed = np.random.randint(2147483647) # make a seed with numpy generator 
        random.seed(seed) # apply this seed to img tranfsorms
        torch.manual_seed(seed) # needed for torchvision 0.7
        if self.img_transform is not None:
            image = self.img_transform(image)
            
        random.seed(seed) # apply this seed to img tranfsorms
        torch.manual_seed(seed) # needed for torchvision 0.7
        if self.gt_transform is not None:
            gt = self.gt_transform(gt)
        return image, gt

    def filter_files(self):
        assert len(self.images) == len(self.gts)
        images = []
        gts = []
        for img_path, gt_path in zip(self.images, self.gts):
            img = Image.open(img_path)
            gt = Image.open(gt_path)
            if img.size == gt.size:
                images.append(img_path)
                gts.append(gt_path)
        self.images = images
        self.gts = gts

    def rgb_loader(self, path):
        with open(path, 'rb') as f:
            img = Image.open(f)
            return img.convert('RGB')

    def binary_loader(self, path):
        with open(path, 'rb') as f:
            img = Image.open(f)
            # return img.convert('1')
            return img.convert('L')

    def resize(self, img, gt):
        assert img.size == gt.size
        w, h = img.size
        if h < self.trainsize or w < self.trainsize:
            h = max(h, self.trainsize)
            w = max(w, self.trainsize)
            return img.resize((w, h), Image.BILINEAR), gt.resize((w, h), Image.NEAREST)
        else:
            return img, gt

    def __len__(self):
        return self.size


class UnlabeledPolypDataset(data.Dataset):
    def __init__(self, image_root, trainsize, augmentations, image_paths=None):
        self.trainsize = trainsize
        self.augmentations = augmentations
        if image_paths is None:
            self.images = [os.path.join(image_root, f) for f in os.listdir(image_root) if _is_image_file(f)]
        else:
            self.images = sorted(list(image_paths))

        self.img_transform = _build_image_transform(self.trainsize, self.augmentations)
        self.size = len(self.images)

    def __getitem__(self, index):
        image = self.rgb_loader(self.images[index])
        if self.img_transform is not None:
            image = self.img_transform(image)
        return image

    def rgb_loader(self, path):
        with open(path, 'rb') as f:
            img = Image.open(f)
            return img.convert('RGB')

    def __len__(self):
        return self.size


def _match_image_mask_pairs(image_root, gt_root):
    image_paths = sorted([os.path.join(image_root, f) for f in os.listdir(image_root) if _is_image_file(f)])
    mask_paths = sorted([os.path.join(gt_root, f) for f in os.listdir(gt_root) if _is_mask_file(f)])

    mask_map = {}
    for m in mask_paths:
        stem = os.path.splitext(os.path.basename(m))[0]
        mask_map[stem] = m

    pairs = []
    for img in image_paths:
        stem = os.path.splitext(os.path.basename(img))[0]
        if stem in mask_map:
            pairs.append((img, mask_map[stem]))

    return pairs


def split_labeled_unlabeled(image_root, gt_root, labeled_ratio=1.0, split_seed=42):
    pairs = _match_image_mask_pairs(image_root, gt_root)
    if len(pairs) == 0:
        return [], [], []

    ratio = float(np.clip(labeled_ratio, 0.0, 1.0))
    indices = np.arange(len(pairs))
    rng = np.random.RandomState(split_seed)
    rng.shuffle(indices)

    if ratio <= 0.0:
        labeled_count = 0
    elif ratio >= 1.0:
        labeled_count = len(pairs)
    else:
        labeled_count = max(1, int(len(pairs) * ratio))

    labeled_idx = set(indices[:labeled_count].tolist())
    labeled_images, labeled_masks, unlabeled_images = [], [], []

    for idx, (img, mask) in enumerate(pairs):
        if idx in labeled_idx:
            labeled_images.append(img)
            labeled_masks.append(mask)
        else:
            unlabeled_images.append(img)

    return labeled_images, labeled_masks, unlabeled_images


def get_loader(image_root, gt_root, batchsize, trainsize, shuffle=True, num_workers=4, pin_memory=True, augmentation=False):

    dataset = PolypDataset(image_root, gt_root, trainsize, augmentation)
    data_loader = data.DataLoader(dataset=dataset,
                                  batch_size=batchsize,
                                  shuffle=shuffle,
                                  num_workers=num_workers,
                                  pin_memory=pin_memory)
    return data_loader


def get_semi_loaders(
    image_root,
    gt_root,
    labeled_batchsize,
    unlabeled_batchsize,
    trainsize,
    labeled_ratio=1.0,
    split_seed=42,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    augmentation=False,
):
    labeled_images, labeled_masks, unlabeled_images = split_labeled_unlabeled(
        image_root, gt_root, labeled_ratio=labeled_ratio, split_seed=split_seed
    )

    labeled_dataset = PolypDataset(
        image_root,
        gt_root,
        trainsize,
        augmentation,
        image_paths=labeled_images,
        gt_paths=labeled_masks,
    )
    labeled_loader = data.DataLoader(
        dataset=labeled_dataset,
        batch_size=labeled_batchsize,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False,
    )

    unlabeled_loader = None
    if len(unlabeled_images) > 0:
        unlabeled_dataset = UnlabeledPolypDataset(
            image_root,
            trainsize,
            augmentation,
            image_paths=unlabeled_images,
        )
        unlabeled_loader = data.DataLoader(
            dataset=unlabeled_dataset,
            batch_size=unlabeled_batchsize,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin_memory,
            drop_last=False,
        )

    split_info = {
        'labeled_count': len(labeled_images),
        'unlabeled_count': len(unlabeled_images),
        'total_count': len(labeled_images) + len(unlabeled_images),
        'labeled_ratio': labeled_ratio,
    }
    return labeled_loader, unlabeled_loader, split_info


class test_dataset:
    def __init__(self, image_root, gt_root, testsize):
        self.testsize = testsize
        self.images = [image_root + f for f in os.listdir(image_root) if f.endswith('.jpg') or f.endswith('.png')]
        self.gts = [gt_root + f for f in os.listdir(gt_root) if f.endswith('.tif') or f.endswith('.png') or f.endswith('.jpg')]
        self.images = sorted(self.images)
        self.gts = sorted(self.gts)
        self.transform = transforms.Compose([
            transforms.Resize((self.testsize, self.testsize)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])])
        self.gt_transform = transforms.ToTensor()
        self.size = len(self.images)
        self.index = 0

    def load_data(self):
        image = self.rgb_loader(self.images[self.index])
        image = self.transform(image).unsqueeze(0)
        gt = self.binary_loader(self.gts[self.index])
        name = self.images[self.index].split('/')[-1]
        if name.endswith('.jpg'):
            name = name.split('.jpg')[0] + '.png'
        self.index += 1
        return image, gt, name

    def rgb_loader(self, path):
        with open(path, 'rb') as f:
            img = Image.open(f)
            return img.convert('RGB')

    def binary_loader(self, path):
        with open(path, 'rb') as f:
            img = Image.open(f)
            return img.convert('L')


Overwriting /kaggle/working/utils/dataloader.py


In [5]:
%%writefile /kaggle/working/train_mean_teacher.sh
python -W ignore train_mean_teacher.py \
  --epoch 100 \
  --optimizer AdamW \
  --lr 1e-4 \
  --batchsize 8 \
  --trainsize 352 \
  --clip 0.5 \
  --decay_rate 0.1 \
  --decay_epoch 50 \
  --augmentation True \
  --num_workers 4 \
  --pin_memory True \
  --use_amp False \
  --labeled_ratio 0.3 \
  --split_seed 42 \
  --unlabeled_batchsize 8 \
  --ema_decay 0.99 \
  --consistency_weight 0.5 \
  --consistency_rampup 30 \
  --student_noise_std 0.08 \
  --teacher_noise_std 0.04

Writing /kaggle/working/train_mean_teacher.sh


In [6]:
# %cd /kaggle/working
# !pip install thop
# !bash train_mean_teacher.sh

In [7]:
%%writefile normal_train.py 
# pyright: reportAttributeAccessIssue=false
import argparse
import logging
import os
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from lib.pvt import PolypPVT
from utils.dataloader import get_semi_loaders, test_dataset
from utils.utils import AvgMeter, adjust_lr, clip_gradient


def structure_loss(pred, mask):
    weit = 1 + 5 * torch.abs(F.avg_pool2d(mask, kernel_size=31, stride=1, padding=15) - mask)
    wbce = F.binary_cross_entropy_with_logits(pred, mask, reduction="none")
    wbce = (weit * wbce).sum(dim=(2, 3)) / weit.sum(dim=(2, 3))

    pred = torch.sigmoid(pred)
    inter = ((pred * mask) * weit).sum(dim=(2, 3))
    union = ((pred + mask) * weit).sum(dim=(2, 3))
    wiou = 1 - (inter + 1) / (union - inter + 1)

    return (wbce + wiou).mean()


@torch.no_grad()
def test(model, path, dataset, test_size):
    data_path = os.path.join(path, dataset)
    image_root = f"{data_path}/images/"
    gt_root = f"{data_path}/masks/"

    model.eval()
    num_samples = len(os.listdir(gt_root))
    loader = test_dataset(image_root, gt_root, test_size)
    dsc_sum = 0.0

    for _ in range(num_samples):
        image, gt, _ = loader.load_data()
        gt = np.asarray(gt, np.float32)
        gt /= (gt.max() + 1e-8)
        image = image.cuda()

        pred1, pred2 = model(image)
        pred = F.interpolate(pred1 + pred2, size=gt.shape, mode="bilinear", align_corners=False)
        pred = pred.sigmoid().data.cpu().numpy().squeeze()
        pred = (pred - pred.min()) / (pred.max() - pred.min() + 1e-8)

        pred_flat = np.reshape(pred, (-1))
        gt_flat = np.reshape(gt, (-1))
        intersection = pred_flat * gt_flat
        smooth = 1
        dice = (2 * intersection.sum() + smooth) / (pred.sum() + gt.sum() + smooth)
        dsc_sum += float(f"{dice:.4f}")

    return dsc_sum / num_samples


def unwrap_model(model):
    return model.module if isinstance(model, torch.nn.parallel.DataParallel) else model


def train_one_epoch(labeled_loader, model, optimizer, scaler, epoch, opt, total_step):
    model.train()

    size_rates = [0.75, 1, 1.25]
    loss_meter = AvgMeter()

    for step, pack in enumerate(labeled_loader, start=1):
        images, gts = pack
        images = images.cuda(non_blocking=True)
        gts = gts.cuda(non_blocking=True)

        for rate in size_rates:
            optimizer.zero_grad()

            train_size = int(round(opt.trainsize * rate / 32) * 32)
            if rate != 1:
                images_r = F.interpolate(images, size=(train_size, train_size), mode="bilinear", align_corners=True)
                gts_r = F.interpolate(gts, size=(train_size, train_size), mode="bilinear", align_corners=True)
            else:
                images_r = images
                gts_r = gts

            with torch.cuda.amp.autocast(enabled=opt.use_amp):
                pred1, pred2 = model(images_r)
                loss1 = structure_loss(pred1, gts_r)
                loss2 = structure_loss(pred2, gts_r)
                total_loss = loss1 + loss2

            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            clip_gradient(optimizer, opt.clip)
            scaler.step(optimizer)
            scaler.update()

            if rate == 1:
                loss_meter.update(total_loss.data, opt.batchsize)

        if step % 20 == 0 or step == total_step:
            print(
                "{} Epoch [{:03d}/{:03d}], Step [{:04d}/{:04d}], loss: {:0.4f}".format(
                    datetime.now(),
                    epoch,
                    opt.epoch,
                    step,
                    total_step,
                    loss_meter.show(),
                )
            )


def validate_and_checkpoint(model, epoch, opt, best_score, dict_plot):
    save_path = opt.train_save
    if not os.path.exists(save_path):
        os.makedirs(save_path)

    model_state = unwrap_model(model).state_dict()
    torch.save(model_state, os.path.join(save_path, f"{epoch}_PolypPVT.pth"))

    test_path = "./dataset/TestDataset/"
    for dataset in ["CVC-300", "CVC-ClinicDB", "Kvasir", "CVC-ColonDB", "ETIS-LaribPolypDB"]:
        dataset_dice = test(model, test_path, dataset, opt.trainsize)
        logging.info("epoch: %s, dataset: %s, dice: %s", epoch, dataset, dataset_dice)
        print(dataset, ": ", dataset_dice)
        dict_plot[dataset].append(dataset_dice)

    mean_dice = test(model, opt.test_path, "test", opt.trainsize)
    dict_plot["test"].append(mean_dice)

    if mean_dice > best_score:
        best_score = mean_dice
        torch.save(model_state, os.path.join(save_path, "PolypPVT_best.pth"))
        torch.save(model_state, os.path.join(save_path, f"{epoch}_PolypPVT_best.pth"))
        print("##############################################################################best", best_score)
        logging.info("##############################################################################best:%s", best_score)

    return best_score


if __name__ == "__main__":
    dict_plot = {
        "CVC-300": [],
        "CVC-ClinicDB": [],
        "Kvasir": [],
        "CVC-ColonDB": [],
        "ETIS-LaribPolypDB": [],
        "test": [],
    }

    model_name = "PolypPVT_LabeledOnly"

    parser = argparse.ArgumentParser()
    parser.add_argument("--epoch", type=int, default=100, help="epoch number")
    parser.add_argument("--lr", type=float, default=1e-4, help="learning rate")
    parser.add_argument("--optimizer", type=str, default="AdamW", help="choosing optimizer AdamW or SGD")
    parser.add_argument("--augmentation", default=False, help="choose to do random flip rotation")
    parser.add_argument("--batchsize", type=int, default=16, help="training batch size")
    parser.add_argument("--trainsize", type=int, default=352, help="training dataset size")
    parser.add_argument("--clip", type=float, default=0.5, help="gradient clipping margin")
    parser.add_argument("--decay_rate", type=float, default=0.1, help="decay rate of learning rate")
    parser.add_argument("--decay_epoch", type=int, default=50, help="every n epochs decay learning rate")
    parser.add_argument("--train_path", type=str, default="./dataset/TrainDataset/", help="path to train dataset")
    parser.add_argument("--test_path", type=str, default="./dataset/TestDataset/", help="path to testing dataset")
    parser.add_argument("--train_save", type=str, default="./model_pth/" + model_name + "/")
    parser.add_argument("--num_workers", type=int, default=4, help="dataloader workers")
    parser.add_argument("--pin_memory", default=True, help="use pin_memory for dataloader")
    parser.add_argument("--labeled_ratio", type=float, default=0.3, help="ratio of training data used as labeled")
    parser.add_argument("--split_seed", type=int, default=42, help="random seed for labeled/unlabeled split")
    parser.add_argument("--use_amp", default=False, help="enable automatic mixed precision")

    opt = parser.parse_args()
    opt.use_amp = str(opt.use_amp).lower() == "true"
    opt.pin_memory = str(opt.pin_memory).lower() == "true"

    torch.backends.cudnn.benchmark = True

    logging.basicConfig(
        filename="train_log.log",
        format="[%(asctime)s-%(filename)s-%(levelname)s:%(message)s]",
        level=logging.INFO,
        filemode="a",
        datefmt="%Y-%m-%d %I:%M:%S %p",
    )

    model = PolypPVT().cuda()

    gpu_count = torch.cuda.device_count()
    if gpu_count > 1:
        print(f"Using DataParallel on {gpu_count} GPUs")
        model = torch.nn.parallel.DataParallel(model)

    if opt.optimizer == "AdamW":
        optimizer = torch.optim.AdamW(model.parameters(), opt.lr, weight_decay=1e-4)
    else:
        optimizer = torch.optim.SGD(model.parameters(), opt.lr, weight_decay=1e-4, momentum=0.9)
    scaler = torch.cuda.amp.GradScaler(enabled=opt.use_amp)

    print(optimizer)

    image_root = f"{opt.train_path}/images/"
    gt_root = f"{opt.train_path}/masks/"

    labeled_loader, _unlabeled_loader, split_info = get_semi_loaders(
        image_root,
        gt_root,
        labeled_batchsize=opt.batchsize,
        unlabeled_batchsize=opt.batchsize,
        trainsize=opt.trainsize,
        labeled_ratio=opt.labeled_ratio,
        split_seed=opt.split_seed,
        num_workers=opt.num_workers,
        pin_memory=opt.pin_memory,
        augmentation=opt.augmentation,
    )

    total_step = len(labeled_loader)

    print(
        "Labeled-only split -> total: {}, labeled: {}, unlabeled: {}, labeled_ratio: {}".format(
            split_info["total_count"],
            split_info["labeled_count"],
            split_info["unlabeled_count"],
            opt.labeled_ratio,
        )
    )

    best = 0.0

    print("#" * 20, "Start Labeled-Only Training", "#" * 20)

    for epoch in range(1, opt.epoch):
        adjust_lr(optimizer, opt.lr, epoch, opt.decay_rate, opt.decay_epoch)
        train_one_epoch(labeled_loader, model, optimizer, scaler, epoch, opt, total_step)
        best = validate_and_checkpoint(model, epoch, opt, best, dict_plot)


Writing normal_train.py


In [8]:
%%writefile normal_train.sh
python -W ignore normal_train.py \
  --epoch 100 \
  --optimizer AdamW \
  --lr 1e-4 \
  --batchsize 8 \
  --trainsize 352 \
  --clip 0.5 \
  --decay_rate 0.1 \
  --decay_epoch 50 \
  --augmentation True \
  --num_workers 4 \
  --pin_memory True \
  --use_amp False \
  --labeled_ratio 0.3 \
  --split_seed 42

Writing normal_train.sh


In [9]:
%cd /kaggle/working
!pip install thop
!bash normal_train.sh

/kaggle/working
AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.0001
)
True
Using RandomRotation, RandomFlip
Labeled-only split -> total: 1450, labeled: 435, unlabeled: 1015, labeled_ratio: 0.3
#################### Start Labeled-Only Training ####################
2026-03-08 11:27:04.730321 Epoch [001/100], Step [0020/0055], loss: 3.1037
2026-03-08 11:27:30.260708 Epoch [001/100], Step [0040/0055], loss: 2.7576
2026-03-08 11:28:13.926055 Epoch [001/100], Step [0055/0055], loss: 2.3349
CVC-300 :  0.7133766666666668
CVC-ClinicDB :  0.6589387096774194
Kvasir :  0.7445930000000002
CVC-ColonDB :  0.5700436842105263
ETIS-LaribPolypDB :  0.5236459183673469
##############################################################################best 0.5982046365914788
2026-03-08 11:29:48.857415 Epoch [0